SSAFY VQA — A100 competition pipeline

이 노트북은 원본 baseline의 치명적인 오류를 수정한 대회용 파이프라인입니다.

- Qwen3.5-9B native VLM + BF16 LoRA (A100 80GB 속도 최적화)
- assistant 정답 token에만 loss 적용
- train/dev/test 공통 prompt adapter
- exact/perceptual image hash 감사와 group-safe split
- 생성 대신 `a/b/c/d` conditional log-probability 채점
- 선택지 순서 permutation ensemble로 위치 편향 완화
- checkpoint 저장/재개, 원자적 추론 저장, 제출 schema 검증

중요: 우승은 보장할 수 없습니다. 이 버전은 Accuracy 평가, 외부 사전학습 모델 및 ensemble 허용이 확인된 대회를 전제로 관련 설정을 기본 `True`로 고정합니다. Public leaderboard는 설정 선택에 사용하지 않습니다.



## 1. Colab 환경 준비

첫 실행 후 런타임을 한 번 재시작하세요. A100은 BF16을 사용하며 FP16 GradScaler를 사용하지 않습니다.



In [ ]:
# Colab이 제공하는 torch/CUDA는 다시 설치하지 않습니다.
# pandas 2.2.3은 google-colab의 고정 요구사항이고 numpy<2.3은 numba와의 호환 조건입니다.
%pip install -q --upgrade-strategy only-if-needed \
  "numpy>=1.26,<2.3" "pandas==2.2.3" \
  "transformers>=5.2.0,<5.3" "accelerate>=1.10.0,<2" "peft>=0.18.0,<0.19" \
  "datasets>=4.0.0,<5" "scikit-learn>=1.6.0,<2" "imagehash>=4.3.2,<5" \
  "huggingface_hub>=0.34.0,<2" "pillow>=10,<12" "safetensors>=0.5,<1"

# 설치 후 반드시 '런타임 > 세션 다시 시작'을 한 번 실행하세요.



## 1-1. Google Drive에서 데이터셋 가져오기

Drive에서 압축이 풀린 `ssafy-16-2-ai` 폴더를 먼저 찾습니다. 폴더가 없고 ZIP만 있으면 비밀번호를 입력받아 `/content`에 풉니다. 비밀번호는 노트북에 저장하지 않습니다.



In [ ]:
from pathlib import Path
from getpass import getpass
import subprocess

MOUNT_GOOGLE_DRIVE = True
AUTO_EXTRACT_DRIVE_ZIP = True
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/ssafy-16-2-ai")
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
DATA_EXTRACT_ROOT = "/content"
_required_csv = ["train.csv", "dev.csv", "test.csv", "sample_submission.csv"]

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if all((DRIVE_DATA_DIR / name).is_file() for name in _required_csv):
    DATA_ROOT_HINT = str(DRIVE_DATA_DIR)
    print("압축 해제된 Drive 데이터셋 사용:", DATA_ROOT_HINT)
elif AUTO_EXTRACT_DRIVE_ZIP and Path(DRIVE_ZIP_PATH).is_file():
    zip_password = getpass("ZIP password: ")
    subprocess.run(["7z", "x", "-y", f"-p{zip_password}", f"-o{DATA_EXTRACT_ROOT}", DRIVE_ZIP_PATH], check=True)
    del zip_password
    DATA_ROOT_HINT = DATA_EXTRACT_ROOT
    print("ZIP 압축 해제 완료:", DATA_EXTRACT_ROOT)
else:
    DATA_ROOT_HINT = DATA_EXTRACT_ROOT
    print("기본 위치와 Drive에서 데이터 폴더를 자동 탐색합니다.")



## 2. 설정

`RUN_MODE`를 단계별로 바꾸어 위에서 아래로 실행합니다.

1. `precheck` → 공식 규칙/metric과 파일 확인
2. `audit` → 이미지 중복 감사와 고정 split 생성
3. `zero_shot` → 학습 전 dev 성능 측정
4. `smoke` → 최대 샘플, 20+5 step 저장/재개 검증
5. `train` → seed 42 split LoRA 학습/검증
6. `validate` → 저장 adapter로 dev 평가
7. `final` → 고정 step 수로 전체 학습 (검증 후에만)
8. `infer` → test 추론
9. `ensemble` → OOF로 검증한 모델 score 평균
10. `submit_check` → sample_submission 기반 제출 생성



In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
from contextlib import nullcontext
from typing import Any, Iterable
import gc, hashlib, importlib.util, json, math, os, random, re, shutil, sys, time, unicodedata

import imagehash
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForMultimodalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, PeftModel, get_peft_model
from huggingface_hub import HfApi

# 기본값: 노트북의 '모두 실행' 한 번으로 데이터 감사/split 생성 후 학습까지 진행
RUN_MODE = "train"
RULES_CONFIRMED = True
OFFICIAL_METRIC = "accuracy"
METRIC_HIGHER_IS_BETTER = True
EXTERNAL_PRETRAINED_ALLOWED = True
ENSEMBLE_ALLOWED = True

# 압축을 /content에 풀면 train.csv, dev.csv, test.csv와 이미지 폴더가 생기는 구조
# 위 데이터 준비 셀의 결과를 사용합니다. 직접 경로를 지정해도 됩니다.
DATA_ROOT = Path(DATA_ROOT_HINT)
AUTO_DETECT_DATA_ROOT = True
WORK_ROOT = Path("/content/vqa_competition")
MODEL_ID = "Qwen/Qwen3.5-9B"
MODEL_REVISION = None  # precheck에서 실제 commit SHA로 고정
SEED = 42
SPLIT_SEEDS = [42, 17, 71]
VALID_RATIO = 0.15

IMAGE_TOKEN_BUDGET = 768  # OCR 중심이라 해상도는 유지
MIN_IMAGE_TOKENS = 256
MAX_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
EPOCHS = 1
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM = 8  # effective batch = 16
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
GRADIENT_CHECKPOINTING = False  # 9B+A100 80GB에서는 속도를 위해 비활성화
TRAIN_VALID_LIMIT = 384  # epoch 중 빠른 checkpoint 비교; 전체 dev 평가는 validate 모드

# 검증된 뒤에만 설정
FINAL_OPTIMIZER_STEPS = None
INCLUDE_DEV_IN_FINAL = False  # 규칙상 dev 학습 사용이 허용되고 모델 선택이 끝난 뒤에만 True
ADAPTER_DIR = WORK_ROOT / "runs" / "best" / "adapter"
INFER_ADAPTER_DIR = ADAPTER_DIR
PREDICTION_FILES_FOR_ENSEMBLE = []
PERMUTATION_PASSES = 2  # 1=최고 속도, 4=최고 위치편향 완화
SAVE_EVERY = 100

for p in [WORK_ROOT / x for x in ["config", "audit", "splits", "runs", "predictions", "submissions"]]:
    p.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available() or RUN_MODE in {"precheck", "audit", "submit_check", "ensemble"}, "GPU가 필요합니다."
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("mode:", RUN_MODE, "torch:", torch.__version__, "cuda:", torch.version.cuda)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0), "BF16:", torch.cuda.is_bf16_supported())



## 3. 데이터 로드와 PRECHECK

Accuracy 평가와 외부 사전학습 모델/ensemble 허용을 확정값으로 사용합니다. dev의 `answer1..answer5`는 다수결 label과 합의도 진단에 사용합니다.



In [ ]:
REQUIRED_DATA_FILES = ["train.csv", "dev.csv", "test.csv", "sample_submission.csv"]

def is_data_root(path: Path) -> bool:
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_DATA_FILES)

def locate_data_root(configured: Path) -> Path:
    configured = Path(configured)
    if is_data_root(configured): return configured
    if not AUTO_DETECT_DATA_ROOT: return configured
    common = [
        Path("/content/ssafy-16-2-ai"),
        Path("/content/drive/MyDrive/ssafy-16-2-ai"),
        Path("/content/drive/MyDrive"),
    ]
    for candidate in common:
        if is_data_root(candidate):
            print(f"DATA_ROOT 자동 탐색: {candidate}")
            return candidate
    drive = Path("/content/drive/MyDrive")
    if drive.exists():
        # 전체 Drive를 무제한 검색하지 않고 최대 깊이 3까지만 확인
        frontier = [drive]
        for _ in range(3):
            next_frontier = []
            for folder in frontier:
                try:
                    children = [x for x in folder.iterdir() if x.is_dir()]
                except (PermissionError, OSError):
                    continue
                for child in children:
                    if is_data_root(child):
                        print(f"DATA_ROOT 자동 탐색: {child}")
                        return child
                next_frontier.extend(children)
            frontier = next_frontier
    raise FileNotFoundError(
        "train.csv/dev.csv/test.csv/sample_submission.csv가 함께 있는 폴더를 찾지 못했습니다. "
        "Drive에 ZIP만 있다면 USE_GOOGLE_DRIVE_ZIP=True로 압축을 먼저 풀고, "
        "이미 풀었다면 DATA_ROOT를 해당 폴더로 지정하세요."
    )

DATA_ROOT = locate_data_root(DATA_ROOT)
TRAIN_CSV = DATA_ROOT / "train.csv"
DEV_CSV = DATA_ROOT / "dev.csv"
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
print("사용할 DATA_ROOT:", DATA_ROOT)

def resolve_image_path(rel: str) -> Path:
    p = Path(str(rel))
    return p if p.is_absolute() else DATA_ROOT / p

def majority_answer(row) -> str:
    vals = [str(row.get(f"answer{i}", "")).strip().lower() for i in range(1, 6)]
    vals = [x for x in vals if x in "abcd"]
    if not vals: raise ValueError(f"dev answer 없음: {row.get('id')}")
    counts = Counter(vals)
    return max("abcd", key=lambda x: (counts[x], -"abcd".index(x)))

def load_data():
    for p in [TRAIN_CSV, DEV_CSV, TEST_CSV, SAMPLE_CSV]:
        if not p.exists(): raise FileNotFoundError(p)
    train = pd.read_csv(TRAIN_CSV)
    dev = pd.read_csv(DEV_CSV).fillna("")
    test = pd.read_csv(TEST_CSV)
    sample = pd.read_csv(SAMPLE_CSV)
    dev["answer"] = dev.apply(majority_answer, axis=1)
    return train, dev, test, sample

train_df, dev_df, test_df, sample_df = load_data()

def precheck():
    required = {
        "train": ({"id","path","question","a","b","c","d","answer"}, train_df),
        "dev": ({"id","path","question","a","b","c","d","answer1","answer2","answer3","answer4","answer5"}, dev_df),
        "test": ({"id","path","question","a","b","c","d"}, test_df),
        "sample": ({"id","answer"}, sample_df),
    }
    for name, (cols, df) in required.items():
        missing = cols - set(df.columns)
        if missing: raise ValueError(f"{name} missing columns: {missing}")
        if df["id"].isna().any() or df["id"].duplicated().any(): raise ValueError(f"{name}: invalid id")
    if len(test_df) != len(sample_df): raise ValueError("test/sample 행 수 불일치")
    if test_df["id"].astype(str).tolist() != sample_df["id"].astype(str).tolist():
        raise ValueError("test/sample ID 또는 순서 불일치")
    for name, df in [("train",train_df),("dev",dev_df),("test",test_df)]:
        empty = df[["question","a","b","c","d"]].isna().any(axis=1)
        if empty.any(): raise ValueError(f"{name}: 필수 text 결측 {empty.sum()}건")
        missing_images = [str(resolve_image_path(x)) for x in df["path"] if not resolve_image_path(x).exists()]
        if missing_images: raise FileNotFoundError(f"{name}: 이미지 {len(missing_images)}개 누락, 예: {missing_images[:3]}")
    if not set(train_df["answer"].astype(str).str.lower()) <= set("abcd"):
        raise ValueError("train answer 범위 오류")
    report = {
        "rows": {"train":len(train_df), "dev":len(dev_df), "test":len(test_df)},
        "columns": {k:list(v[1].columns) for k,v in required.items()},
        "answer_distribution": train_df["answer"].value_counts().sort_index().to_dict(),
        "official_metric": OFFICIAL_METRIC,
        "rules_confirmed": RULES_CONFIRMED,
        "external_pretrained_allowed": EXTERNAL_PRETRAINED_ALLOWED,
        "ensemble_allowed": ENSEMBLE_ALLOWED,
    }
    (WORK_ROOT/"config"/"precheck.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    if RUN_MODE not in {"precheck", "audit"}:
        if not RULES_CONFIRMED: raise ValueError("공식 규칙/metric 확인 후 RULES_CONFIRMED=True로 설정하세요.")
        if EXTERNAL_PRETRAINED_ALLOWED is not True: raise ValueError("외부 pretrained 모델 허용 여부를 확인하세요.")
        if OFFICIAL_METRIC is None or METRIC_HIGHER_IS_BETTER is None: raise ValueError("공식 metric 설정 필요")

precheck()



## 4. 데이터 감사와 leakage-safe split

SHA-256 동일 이미지와 pHash Hamming distance ≤ 4인 이미지를 transitive group으로 묶습니다. 감사 결과가 없으면 이후 학습은 중단합니다.



In [ ]:
class UnionFind:
    def __init__(self, n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]: a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]: self.r[a]+=1

class BKNode:
    def __init__(self, value, idx): self.value=value; self.indices=[idx]; self.children={}
    def add(self, value, idx):
        d=(self.value^value).bit_count()
        if d==0: self.indices.append(idx)
        elif d in self.children: self.children[d].add(value,idx)
        else: self.children[d]=BKNode(value,idx)
    def query(self, value, radius, out):
        d=(self.value^value).bit_count()
        if d<=radius: out.extend(self.indices)
        for k,ch in self.children.items():
            if d-radius<=k<=d+radius: ch.query(value,radius,out)

def hash_one(path: Path):
    raw = path.read_bytes()
    sha = hashlib.sha256(raw).hexdigest()
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        w,h=im.size
        ph=int(str(imagehash.phash(im)),16)
    return sha, ph, w, h, len(raw)

def run_audit_and_splits():
    parts=[]
    for source,df in [("train",train_df),("dev",dev_df),("test",test_df)]:
        x=df[["id","path","question"]].copy(); x["source"]=source; parts.append(x)
    all_df=pd.concat(parts,ignore_index=True)
    records=[]
    for r in tqdm(all_df.itertuples(index=False), total=len(all_df), desc="image audit"):
        p=resolve_image_path(r.path)
        try:
            sha,ph,w,h,n=hash_one(p); err=""
        except Exception as e:
            sha=ph=w=h=n=None; err=repr(e)
        records.append({"id":r.id,"source":r.source,"path":r.path,"sha256":sha,"phash":ph,
                        "width":w,"height":h,"bytes":n,"decode_error":err})
    audit=pd.DataFrame(records)
    if (audit.decode_error!="").any():
        audit.to_csv(WORK_ROOT/"audit"/"image_audit.csv",index=False)
        raise ValueError("decode 불가 이미지가 있습니다.")
    uf=UnionFind(len(audit)); sha_first={}
    for i,s in enumerate(audit.sha256):
        if s in sha_first: uf.union(i,sha_first[s])
        else: sha_first[s]=i
    root=None
    for i,v in tqdm(enumerate(audit.phash.astype(object)), total=len(audit), desc="near duplicate groups"):
        v=int(v); hits=[]
        if root is None: root=BKNode(v,i)
        else:
            root.query(v,4,hits)
            for j in hits: uf.union(i,j)
            root.add(v,i)
    roots=[uf.find(i) for i in range(len(audit))]
    root_to_gid={r:f"g{j:06d}" for j,r in enumerate(sorted(set(roots)))}
    audit["image_group_id"]=[root_to_gid[r] for r in roots]
    audit.to_csv(WORK_ROOT/"audit"/"image_audit.csv",index=False)
    dup=audit[audit.duplicated("image_group_id",False)].sort_values("image_group_id")
    dup.to_csv(WORK_ROOT/"audit"/"duplicate_groups.csv",index=False)
    cross=(dup.groupby("image_group_id")["source"].nunique()>1)
    report={
        "images":len(audit), "groups":audit.image_group_id.nunique(),
        "duplicate_rows":len(dup), "cross_source_groups":int(cross.sum()),
        "train_exact_duplicate_ids":int(train_df.id.duplicated().sum()),
        "normalized_question_duplicates":int(pd.concat(parts).question.astype(str).map(
            lambda x:unicodedata.normalize("NFKC",x).strip().lower()).duplicated().sum()),
    }
    (WORK_ROOT/"audit"/"data_audit.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
    train_groups=audit[audit.source=="train"].set_index("id").loc[train_df.id,"image_group_id"].values
    n_splits=max(2,round(1/VALID_RATIO))
    for seed in SPLIT_SEEDS:
        cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
        tr_idx,va_idx=next(cv.split(train_df,train_df.answer,groups=train_groups))
        split=pd.DataFrame({"id":train_df.id,"image_group_id":train_groups,"split":"train","seed":seed})
        split.loc[va_idx,"split"]="valid"
        assert not (set(split.loc[split.split=="train","image_group_id"]) & set(split.loc[split.split=="valid","image_group_id"]))
        path=WORK_ROOT/"splits"/f"split_seed{seed}.csv"; split.to_csv(path,index=False)
        print(path, split.split.value_counts().to_dict(), hashlib.sha256(path.read_bytes()).hexdigest())
    print(json.dumps(report,ensure_ascii=False,indent=2))

if RUN_MODE=="audit":
    run_audit_and_splits()
elif RUN_MODE in {"smoke", "train"} and not (WORK_ROOT/"splits"/f"split_seed{SEED}.csv").exists():
    print("고정 split이 없어 데이터 감사와 split 생성을 먼저 실행합니다.")
    run_audit_and_splits()



## 5. 공통 prompt와 데이터셋

train/validation/test 모두 같은 `build_messages()`를 사용합니다. 선택지 permutation은 정답 위치 shortcut을 줄이고, 추론 때는 원래 선택지 score로 다시 합칩니다.



In [ ]:
SYSTEM_PROMPT = "You are a precise visual question answering system. Inspect the image carefully, including small text."
LABELS = list("abcd")

def normalize_answer(x: str) -> str:
    x=unicodedata.normalize("NFKC",str(x)).strip().lower()
    x=re.sub(r"<think>.*?</think>","",x,flags=re.S).strip()
    x=re.sub(r"^(answer|정답)\s*[:：]?\s*","",x).strip()
    m=re.search(r"(?<![a-z])[abcd](?![a-z])",x)
    return m.group(0) if m else ""

def permute_row(row, perm=(0,1,2,3)):
    choices=[str(row[x]) for x in LABELS]
    gold=str(row.get("answer","")).lower()
    old_gold=LABELS.index(gold) if gold in LABELS else None
    mapped=[choices[i] for i in perm]
    new_gold=LABELS[perm.index(old_gold)] if old_gold is not None else ""
    return mapped,new_gold

def build_messages(row, image, perm=(0,1,2,3), include_answer=False):
    choices,gold=permute_row(row,perm)
    prompt=(f"{row['question']}\n" + "\n".join(f"({LABELS[i]}) {choices[i]}" for i in range(4)) +
            "\n\n가장 알맞은 답 하나를 고르세요. a, b, c, d 중 소문자 한 글자만 출력하세요.")
    msgs=[
        {"role":"system","content":[{"type":"text","text":SYSTEM_PROMPT}]},
        {"role":"user","content":[{"type":"image","image":image},{"type":"text","text":prompt}]},
    ]
    if include_answer: msgs.append({"role":"assistant","content":[{"type":"text","text":gold}]})
    return msgs

class VQADataset(Dataset):
    def __init__(self, df, training=False, permute=True, seed=42):
        self.df=df.reset_index(drop=True); self.training=training; self.permute=permute; self.seed=seed; self.epoch=0
    def set_epoch(self,e): self.epoch=e
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i].to_dict()
        with Image.open(resolve_image_path(row["path"])) as im:
            image=ImageOps.exif_transpose(im).convert("RGB")
        perm=(0,1,2,3)
        if self.training and self.permute:
            rng=random.Random(self.seed+self.epoch*1_000_003+i); perm=list(range(4)); rng.shuffle(perm); perm=tuple(perm)
        return {"row":row,"image":image,"perm":perm}

@dataclass
class AnswerOnlyCollator:
    processor: Any
    max_length: int = MAX_LENGTH
    def __call__(self,batch):
        kw={"tokenize":False,"enable_thinking":False}
        prompt_texts=[]; full_texts=[]; images=[]; metadata=[]
        for s in batch:
            row,image,perm=s["row"],s["image"],s["perm"]
            prompt_msgs=build_messages(row,image,perm,include_answer=False)
            full_msgs=build_messages(row,image,perm,include_answer=True)
            prompt_texts.append(self.processor.apply_chat_template(prompt_msgs,add_generation_prompt=True,**kw))
            full_texts.append(self.processor.apply_chat_template(full_msgs,add_generation_prompt=False,**kw))
            images.append(image); metadata.append((row,perm))
        prompt=self.processor(text=prompt_texts,images=images,padding=True,truncation=True,
                              max_length=self.max_length,return_tensors="pt")
        full=self.processor(text=full_texts,images=images,padding=True,truncation=True,
                            max_length=self.max_length,return_tensors="pt")
        labels=full["input_ids"].clone(); labels[full["attention_mask"]==0]=-100
        for i,(row,perm) in enumerate(metadata):
            plen=int(prompt["attention_mask"][i].sum()); flen=int(full["attention_mask"][i].sum())
            if flen<=plen: raise ValueError("정답 token이 truncation되었습니다. MAX_LENGTH를 늘리세요.")
            if not torch.equal(prompt["input_ids"][i,:plen],full["input_ids"][i,:plen]):
                raise ValueError("assistant 경계 prefix 불일치 — chat template/model revision을 확인하세요.")
            labels[i,:plen]=-100
            active=labels[i]!=-100
            if int(active.sum())==0: raise ValueError("active answer token 0개")
            decoded=self.processor.tokenizer.decode(full["input_ids"][i][active])
            _,gold=permute_row(row,perm)
            if gold not in normalize_answer(decoded): raise ValueError(f"loss mask 오류: gold={gold}, target={decoded!r}")
        full["labels"]=labels
        return full



## 6. 모델/LoRA 로드와 구조 검사

vision encoder는 동결합니다. 실제 language linear module 이름을 먼저 수집하고, 존재하는 module만 LoRA 대상으로 지정합니다.



In [ ]:
def resolve_revision(model_id=MODEL_ID, revision=MODEL_REVISION):
    if revision: return revision
    return HfApi().model_info(model_id,revision="main").sha

def load_base_model(for_training=False):
    revision=resolve_revision()
    max_pixels=IMAGE_TOKEN_BUDGET*28*28; min_pixels=MIN_IMAGE_TOKENS*28*28
    processor=AutoProcessor.from_pretrained(MODEL_ID,revision=revision,min_pixels=min_pixels,max_pixels=max_pixels)
    processor.tokenizer.padding_side="right"
    attention_backend="flash_attention_2" if importlib.util.find_spec("flash_attn") else "sdpa"
    print("attention backend:",attention_backend)
    model=AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,revision=revision,dtype=torch.bfloat16,
        attn_implementation=attention_backend,device_map="cuda")
    if for_training:
        model.config.use_cache=False
        if hasattr(model.config,"text_config"): model.config.text_config.use_cache=False
        if GRADIENT_CHECKPOINTING:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
            if hasattr(model,"enable_input_require_grads"): model.enable_input_require_grads()
    meta={"model_id":MODEL_ID,"model_revision":revision,"model_class":model.__class__.__name__,
          "processor_class":processor.__class__.__name__,"image_token_budget":IMAGE_TOKEN_BUDGET}
    (WORK_ROOT/"config"/"model.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")
    print(meta)
    return model,processor,revision

def add_lora(model):
    leaf_allow={"q_proj","k_proj","v_proj","o_proj",
                "in_proj_qkvz","in_proj_ba","out_proj"}
    targets=[]
    for name,module in model.named_modules():
        low=name.lower(); leaf=name.rsplit(".",1)[-1]
        if isinstance(module,torch.nn.Linear) and leaf in leaf_allow and not any(x in low for x in ["visual","vision"]):
            targets.append(name)
    if not targets: raise ValueError("LoRA target module을 찾지 못했습니다. 모델 구조를 확인하세요.")
    cfg=LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=LORA_DROPOUT,bias="none",
                   target_modules=targets,task_type="CAUSAL_LM")
    model=get_peft_model(model,cfg)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())
    vision_trainable=[n for n,p in model.named_parameters() if p.requires_grad and any(x in n.lower() for x in ["visual","vision"])]
    if vision_trainable: raise ValueError(f"vision parameter가 trainable입니다: {vision_trainable[:5]}")
    print(f"LoRA modules={len(targets)}, trainable={trainable:,}/{total:,} ({100*trainable/total:.4f}%)")
    print("target examples:",targets[:20])
    return model,targets



## 7. 후보 log-probability 추론

생성/파싱 대신 각 label continuation의 조건부 log-prob를 직접 비교합니다. 네 번의 cyclic choice permutation score를 원래 선택지로 환산해 평균하면 위치 편향이 줄어듭니다.



In [ ]:
CYCLIC_PERMS=[(0,1,2,3),(1,2,3,0),(2,3,0,1),(3,0,1,2)]

@torch.inference_mode()
def score_one_permutation(model,processor,row,image,perm=(0,1,2,3)):
    msgs=build_messages(row,image,perm,include_answer=False)
    prompt_text=processor.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    p=processor(text=[prompt_text],images=[image],return_tensors="pt")
    plen=int(p["attention_mask"][0].sum())
    texts=[prompt_text+x for x in LABELS]
    enc=processor(text=texts,images=[image]*4,padding=True,return_tensors="pt").to(model.device)
    for i in range(4):
        if not torch.equal(p["input_ids"][0,:plen],enc["input_ids"][i,:plen].cpu()):
            raise ValueError("후보 continuation prefix 불일치")
    with torch.autocast("cuda",dtype=torch.bfloat16): out=model(**enc)
    logp=out.logits.float().log_softmax(-1)
    new_scores=[]
    for i in range(4):
        end=int(enc["attention_mask"][i].sum())
        ids=enc["input_ids"][i,plen:end]
        if len(ids)==0: raise ValueError("후보 token 0개")
        positions=torch.arange(plen-1,end-1,device=ids.device)
        tok_logp=logp[i,positions,ids]
        new_scores.append(float(tok_logp.mean().cpu()))
    original=np.full(4,-np.inf,dtype=np.float64)
    for new_pos,old_pos in enumerate(perm): original[old_pos]=new_scores[new_pos]
    return original

@torch.inference_mode()
def predict_one(model,processor,row,permutation_passes=PERMUTATION_PASSES):
    with Image.open(resolve_image_path(row["path"])) as im:
        image=ImageOps.exif_transpose(im).convert("RGB")
    permutation_passes=int(permutation_passes)
    if not 1<=permutation_passes<=4: raise ValueError("permutation_passes는 1~4")
    perms=CYCLIC_PERMS[:permutation_passes]
    scores=np.stack([score_one_permutation(model,processor,row,image,p) for p in perms]).mean(0)
    pred=LABELS[int(scores.argmax())]
    probs=np.exp(scores-scores.max()); probs/=probs.sum()
    return pred,scores,float(probs.max())

def dev_metrics(pred_df, reference=dev_df):
    x=reference.set_index("id").loc[pred_df.id]
    pred=pred_df.prediction.values
    majority=(pred==x.answer.values).mean()
    soft=[]
    for j,p in enumerate(pred):
        votes=sum(str(x.iloc[j][f"answer{i}"]).lower()==p for i in range(1,6))
        soft.append(min(votes/3,1.0))
    return {"accuracy_majority":float(majority),"vqa_soft_diagnostic":float(np.mean(soft)),"n":len(pred)}

def predict_frame(model,processor,df,out_path=None,limit=None,permutation_passes=PERMUTATION_PASSES):
    model.eval(); rows=[]; use=df if limit is None else df.iloc[:limit]
    for row in tqdm(use.to_dict("records"),desc="scoring"):
        pred,scores,conf=predict_one(model,processor,row,permutation_passes=permutation_passes)
        rows.append({"id":row["id"],"prediction":pred,"confidence":conf,
                     **{f"score_{x}":scores[i] for i,x in enumerate(LABELS)}})
    out=pd.DataFrame(rows)
    if out_path: out.to_csv(out_path,index=False)
    return out



## 8. zero-shot / loss-mask 검사



In [ ]:
if RUN_MODE=="zero_shot":
    model,processor,revision=load_base_model(False)
    preds=predict_frame(model,processor,dev_df,WORK_ROOT/"runs"/"zero_shot_dev.csv")
    metrics=dev_metrics(preds); print(metrics)
    (WORK_ROOT/"runs"/"zero_shot_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")

def verify_loss_masks(processor,df=train_df,n=3):
    collator=AnswerOnlyCollator(processor)
    ds=VQADataset(df.iloc[:n],training=True)
    for i in range(n):
        batch=collator([ds[i]])
        active=batch["labels"]!=-100
        assert active.sum()>0 and not torch.equal(batch["labels"],batch["input_ids"])
        print(i,processor.tokenizer.decode(batch["input_ids"][active]))



## 9. 학습, checkpoint, 재개

validation loss가 아니라 후보 log-prob accuracy로 best adapter를 선택합니다. 마지막 불완전 accumulation도 optimizer step에 반영합니다.



In [ ]:
def get_split(seed=42):
    path=WORK_ROOT/"splits"/f"split_seed{seed}.csv"
    if not path.exists(): raise FileNotFoundError("먼저 RUN_MODE='audit'을 실행하세요.")
    split=pd.read_csv(path).set_index("id")
    tr=train_df[train_df.id.map(split.split)=="train"].reset_index(drop=True)
    va=train_df[train_df.id.map(split.split)=="valid"].reset_index(drop=True)
    return tr,va,hashlib.sha256(path.read_bytes()).hexdigest()

def save_checkpoint(model,processor,optimizer,scheduler,state,run_dir):
    tmp=run_dir/"checkpoint_tmp"; final=run_dir/"checkpoint"
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    model.save_pretrained(tmp/"adapter"); processor.save_pretrained(tmp/"adapter")
    torch.save({"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),"state":state},tmp/"trainer_state.pt")
    if final.exists(): shutil.rmtree(final)
    tmp.rename(final)

def train_model(model,processor,train_data,valid_data,run_dir,epochs=EPOCHS,max_optimizer_steps=None,resume=False):
    run_dir.mkdir(parents=True,exist_ok=True)
    ds=VQADataset(train_data,training=True,permute=True,seed=SEED)
    loader=DataLoader(ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,
                      collate_fn=AnswerOnlyCollator(processor),generator=torch.Generator().manual_seed(SEED))
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    planned=max_optimizer_steps or epochs*math.ceil(len(loader)/GRAD_ACCUM)
    scheduler=get_cosine_schedule_with_warmup(optimizer,max(1,int(planned*WARMUP_RATIO)),planned)
    state={"epoch":0,"global_step":0,"best_metric":-1.0,"samples_seen":0}
    ckpt=run_dir/"checkpoint"
    if resume and (ckpt/"trainer_state.pt").exists():
        saved=torch.load(ckpt/"trainer_state.pt",map_location="cpu",weights_only=False)
        optimizer.load_state_dict(saved["optimizer"]); scheduler.load_state_dict(saved["scheduler"]); state=saved["state"]
        print("resumed:",state)
    model.train(); optimizer.zero_grad(set_to_none=True); torch.cuda.reset_peak_memory_stats()
    stop=False
    for epoch in range(state["epoch"],epochs):
        ds.set_epoch(epoch); running=0.0
        bar=tqdm(loader,desc=f"epoch {epoch+1}")
        for micro,batch in enumerate(bar,1):
            batch={k:v.to(model.device,non_blocking=True) for k,v in batch.items()}
            group_start=((micro-1)//GRAD_ACCUM)*GRAD_ACCUM
            group_size=min(GRAD_ACCUM,len(loader)-group_start)
            with torch.autocast("cuda",dtype=torch.bfloat16): loss=model(**batch).loss/group_size
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite loss: {loss}")
            loss.backward(); running+=float(loss.detach())*group_size; state["samples_seen"]+=len(batch["input_ids"])
            should_step=(micro%GRAD_ACCUM==0) or (micro==len(loader))
            if should_step:
                grad=torch.nn.utils.clip_grad_norm_(params,MAX_GRAD_NORM)
                if not torch.isfinite(grad): raise FloatingPointError(f"non-finite grad: {grad}")
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True); state["global_step"]+=1
                bar.set_postfix(loss=f"{running/group_size:.4f}",step=state["global_step"]); running=0.0
                if max_optimizer_steps and state["global_step"]>=max_optimizer_steps: stop=True; break
        state["epoch"]=epoch+1
        save_checkpoint(model,processor,optimizer,scheduler,state,run_dir)
        if valid_data is not None and len(valid_data):
            pred_path=run_dir/f"valid_epoch{epoch+1}.csv"
            eval_data=valid_data
            if TRAIN_VALID_LIMIT and len(eval_data)>TRAIN_VALID_LIMIT:
                per_class=max(1,TRAIN_VALID_LIMIT//4)
                eval_data=pd.concat([g.sample(n=min(len(g),per_class),random_state=SEED)
                                     for _,g in eval_data.groupby("answer")]).reset_index(drop=True)
            vp=predict_frame(model,processor,eval_data,pred_path,permutation_passes=1)
            metric=float((vp.prediction.values==eval_data.answer.values).mean())
            print("valid accuracy:",metric)
            if metric>state["best_metric"]:
                state["best_metric"]=metric
                best=run_dir/"best"; tmp=run_dir/"best_tmp"
                if tmp.exists(): shutil.rmtree(tmp)
                model.save_pretrained(tmp); processor.save_pretrained(tmp)
                if best.exists(): shutil.rmtree(best)
                tmp.rename(best)
        if stop: break
    peak=torch.cuda.max_memory_reserved()/2**30
    if peak>torch.cuda.get_device_properties(0).total_memory/2**30*0.94:
        raise MemoryError(f"peak reserved {peak:.1f}GB > 94%: 본 학습 설정을 축소하세요.")
    metrics={**state,"peak_reserved_vram_gb":peak,"planned_steps":planned}
    (run_dir/"metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
    return metrics

def fresh_trainable_model():
    base,processor,revision=load_base_model(True)
    model,targets=add_lora(base)
    return model,processor,revision,targets

if RUN_MODE in {"smoke","train"}:
    tr,va,split_hash=get_split(SEED)
    model,processor,revision,targets=fresh_trainable_model()
    verify_loss_masks(processor)
    if RUN_MODE=="smoke":
        # 큰 이미지·긴 질문 우선으로 20 optimizer steps
        sizes=train_df.question.astype(str).str.len()+train_df[LABELS].astype(str).apply(lambda x:x.str.len()).sum(axis=1)
        smoke=train_df.loc[sizes.nlargest(min(len(train_df),GRAD_ACCUM*20)).index].reset_index(drop=True)
        run_dir=WORK_ROOT/"runs"/"smoke"
        train_model(model,processor,smoke,None,run_dir,epochs=99,max_optimizer_steps=20)
        del model; gc.collect(); torch.cuda.empty_cache()
        base,processor,_=load_base_model(True)
        model=PeftModel.from_pretrained(base,run_dir/"checkpoint"/"adapter",is_trainable=True)
        train_model(model,processor,smoke,None,run_dir,epochs=99,max_optimizer_steps=25,resume=True)
        temp=predict_frame(model,processor,va.iloc[:32],run_dir/"valid32.csv")
        test16=predict_frame(model,processor,test_df.iloc[:16],run_dir/"test16.csv")
        assert len(temp)==32 and len(test16)==16
    else:
        model_tag=MODEL_ID.rsplit("/",1)[-1].lower().replace(".","").replace("-","_")
        run_id=f"{model_tag}_seed{SEED}_r{LORA_R}_img{IMAGE_TOKEN_BUDGET}"
        run_dir=WORK_ROOT/"runs"/run_id
        cfg={"run_id":run_id,"model_id":MODEL_ID,"revision":revision,"split_hash":split_hash,"seed":SEED,
             "lr":LEARNING_RATE,"rank":LORA_R,"image_tokens":IMAGE_TOKEN_BUDGET,"targets":targets}
        (run_dir).mkdir(parents=True,exist_ok=True)
        (run_dir/"config.json").write_text(json.dumps(cfg,indent=2),encoding="utf-8")
        metrics=train_model(model,processor,tr,va,run_dir)
        # 명시적 best 경로로 복사 (기존 결과는 보존)
        ADAPTER_DIR.parent.mkdir(parents=True,exist_ok=True)
        if ADAPTER_DIR.exists(): shutil.rmtree(ADAPTER_DIR)
        shutil.copytree(run_dir/"best",ADAPTER_DIR)



## 10. 저장 adapter 검증 / 최종 전체 학습

`final`은 validation에서 선택된 optimizer step 수를 `FINAL_OPTIMIZER_STEPS`에 고정한 뒤 pretrained base부터 새로 시작합니다.



In [ ]:
def load_adapter(adapter_dir,trainable=False):
    base,processor,revision=load_base_model(trainable)
    model=PeftModel.from_pretrained(base,adapter_dir,is_trainable=trainable)
    return model,processor,revision

if RUN_MODE=="validate":
    model,processor,revision=load_adapter(INFER_ADAPTER_DIR,False)
    preds=predict_frame(model,processor,dev_df,WORK_ROOT/"runs"/"dev_predictions.csv")
    metrics=dev_metrics(preds); print(metrics)
    (WORK_ROOT/"runs"/"dev_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")

if RUN_MODE=="final":
    if FINAL_OPTIMIZER_STEPS is None: raise ValueError("검증에서 고정한 FINAL_OPTIMIZER_STEPS를 입력하세요.")
    full=train_df.copy()
    if INCLUDE_DEV_IN_FINAL:
        full=pd.concat([full,dev_df[train_df.columns]],ignore_index=True)
    model,processor,revision,targets=fresh_trainable_model()  # adapter 이어학습 금지
    run_dir=WORK_ROOT/"runs"/f"final_steps{FINAL_OPTIMIZER_STEPS}"
    train_model(model,processor,full,None,run_dir,epochs=999,max_optimizer_steps=FINAL_OPTIMIZER_STEPS)
    print("final adapter:",run_dir/"checkpoint"/"adapter")



## 11. test 추론 — 중간 저장 및 재개

100개마다 임시 파일을 원자적으로 교체합니다. 오류가 한 건이라도 있으면 제출 생성을 막습니다.



In [ ]:
def atomic_csv(df,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp")
    df.to_csv(tmp,index=False); os.replace(tmp,path)

def infer_resumable(model,processor,df,out_path):
    out_path=Path(out_path); rows=[]; done=set(); errors=[]
    if out_path.exists():
        old=pd.read_csv(out_path); rows=old.to_dict("records"); done=set(old.id.astype(str)); print("resume",len(done))
    for row in tqdm(df.to_dict("records"),desc="test inference"):
        if str(row["id"]) in done: continue
        try:
            pred,scores,conf=predict_one(model,processor,row)
            if pred not in LABELS: raise ValueError(f"invalid prediction {pred!r}")
            rows.append({"id":row["id"],"prediction":pred,"confidence":conf,
                         **{f"score_{x}":scores[i] for i,x in enumerate(LABELS)}})
        except Exception as e:
            errors.append({"id":row["id"],"error":repr(e)})
        if (len(rows)+len(errors))%SAVE_EVERY==0:
            atomic_csv(pd.DataFrame(rows),out_path)
            if errors: atomic_csv(pd.DataFrame(errors),out_path.with_name(out_path.stem+"_errors.csv"))
    atomic_csv(pd.DataFrame(rows),out_path)
    if errors:
        atomic_csv(pd.DataFrame(errors),out_path.with_name(out_path.stem+"_errors.csv"))
        raise RuntimeError(f"추론 오류 {len(errors)}건 — 제출 생성을 중단합니다.")
    pred=pd.DataFrame(rows).set_index("id").loc[df.id.astype(str)].reset_index()
    if len(pred)!=len(df) or pred.id.duplicated().any(): raise ValueError("추론 누락/중복")
    return pred

if RUN_MODE=="infer":
    model,processor,revision=load_adapter(INFER_ADAPTER_DIR,False)
    out=WORK_ROOT/"predictions"/f"test_{Path(INFER_ADAPTER_DIR).parent.name}.csv"
    infer_resumable(model,processor,test_df,out)
    print(out)



## 12. score ensemble과 최종 제출 검증

ensemble weight는 반드시 OOF/dev에서 먼저 고정하세요. 단순 평균이 단일 모델보다 좋지 않으면 ensemble을 제출하지 않습니다.



In [ ]:
def ensemble_scores(paths,out_path):
    if len(paths)<2: raise ValueError("검증된 prediction 파일을 2개 이상 지정하세요.")
    frames=[pd.read_csv(p).set_index("id") for p in paths]
    ids=frames[0].index
    if any(not ids.equals(x.index) for x in frames[1:]): raise ValueError("ensemble ID/순서 불일치")
    scores=np.mean([x[[f"score_{c}" for c in LABELS]].values for x in frames],axis=0)
    out=pd.DataFrame({"id":ids,"prediction":[LABELS[i] for i in scores.argmax(1)]})
    for i,c in enumerate(LABELS): out[f"score_{c}"]=scores[:,i]
    atomic_csv(out,out_path); return out

if RUN_MODE=="ensemble":
    if ENSEMBLE_ALLOWED is not True: raise ValueError("대회 규칙에서 ensemble 허용 여부를 확인하세요.")
    ensemble_scores(PREDICTION_FILES_FOR_ENSEMBLE,WORK_ROOT/"predictions"/"test_ensemble.csv")

def create_submission(prediction_csv,submission_path):
    pred=pd.read_csv(prediction_csv)
    pred_col="prediction" if "prediction" in pred.columns else "answer"
    if pred.id.duplicated().any() or pred[pred_col].isna().any(): raise ValueError("prediction 중복/결측")
    if not set(pred[pred_col].astype(str))<=set(LABELS): raise ValueError("허용 범위 밖 답")
    if pred.id.astype(str).tolist()!=test_df.id.astype(str).tolist(): raise ValueError("test ID/순서 불일치")
    sub=sample_df.copy()
    if sub.columns.tolist()!=["id","answer"]: raise ValueError(f"sample schema 확인 필요: {sub.columns.tolist()}")
    sub["answer"]=pred[pred_col].values
    if len(sub)!=len(test_df) or sub.answer.isna().any() or (sub.answer.astype(str).str.len()==0).any():
        raise ValueError("submission 행/답 오류")
    atomic_csv(sub,submission_path)
    digest=hashlib.sha256(Path(submission_path).read_bytes()).hexdigest()
    print("VALID submission:",submission_path,"sha256:",digest)
    return sub

if RUN_MODE=="submit_check":
    # 아래 경로를 검증된 단일 모델 또는 OOF 개선이 확인된 ensemble 파일로 지정
    PREDICTION_CSV=WORK_ROOT/"predictions"/"test_best.csv"
    create_submission(PREDICTION_CSV,WORK_ROOT/"submissions"/"submission_best.csv")



## 13. 실험 의사결정 체크리스트

- zero-shot → image token 512/768/1024 → LR 2e-5/5e-5/1e-4 → rank 16/32 순서로 한 축씩 비교
- 최종 후보만 seed 17/42/71에서 비교하고 `평균 - 0.5 × 표준편차`로 선택
- dev majority accuracy, VQA soft diagnostic, 유형별/OCR별 성능과 파싱 실패를 함께 기록
- public leaderboard 변화만으로 model/hyperparameter/ensemble weight를 바꾸지 않음
- 최종 제출은 검증 최강 단일 모델과 OOF 개선이 확인된 보수적 ensemble 최대 2개
